# EDA — Datathon Dự báo Doanh số 2026

Câu chuyện: **"Bộ máy mùa vụ vận hành tốt + Bộ máy biên lợi nhuận hỏng"**

Bốn hồi: Suy thoái → Bộ máy mùa vụ → Rò rỉ biên lợi nhuận → Điểm neo dự báo

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Đường dẫn tuyệt đối — không bị ảnh hưởng khi VS Code / JupyterLab đổi thư mục
_ROOT   = "/Users/xps/Documents/PROJECTS/datathon/datathon-2026"
DATA    = f"{_ROOT}/data/"
OUTPUTS = f"{_ROOT}/outputs/"
os.makedirs(OUTPUTS, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.autolayout": True})

def savefig(name):
    path = os.path.join(OUTPUTS, name)
    plt.savefig(path, bbox_inches="tight")
    print(f"saved → {path}")
    plt.show()

print(f"DATA    = {DATA}")
print(f"OUTPUTS = {OUTPUTS}")
print("Khởi tạo hoàn tất ✓")


---
## Hồi 1 — Suy thoái

In [ ]:
# ── Đọc dữ liệu bán hàng + tính tổng hàng năm (dùng chung cho các hồi) ────────────
sales = pd.read_csv(f"{DATA}sales.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
sales["Year"]  = sales["Date"].dt.year
sales["Month"] = sales["Date"].dt.month

annual = (
    sales.groupby("Year")
         .agg(Revenue=("Revenue", "sum"), COGS=("COGS", "sum"))
         .reset_index()
)
annual["Margin"] = (annual["Revenue"] - annual["COGS"]) / annual["Revenue"] * 100

print(annual.to_string(index=False))


In [ ]:
# ── 1a: Biểu đồ cột doanh thu theo năm — màu theo giai đoạn + đường biên lợi nhuận ─────────
ERA_COLOR = {
    2012: "steelblue", 2013: "steelblue", 2014: "steelblue",
    2015: "steelblue", 2016: "steelblue",
    2017: "darkorange", 2018: "darkorange",
    2019: "gray",       2020: "gray",       2021: "gray", 2022: "gray",
}
colors = [ERA_COLOR[y] for y in annual["Year"]]

fig, ax1 = plt.subplots(figsize=(13, 6))

bars = ax1.bar(annual["Year"], annual["Revenue"] / 1e9, color=colors, alpha=0.85, zorder=2)
ax1.set_xlabel("Năm")
ax1.set_ylabel("Doanh thu (Tỷ VND)")
ax1.set_title(
    "Doanh thu hàng năm 2012–2022  |  xanh = Tăng trưởng  ·  cam = Suy giảm  ·  xám = Bình ổn",
    fontsize=11
)
ax1.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}B"))
ax1.set_ylim(0, 2.75)

for bar, val in zip(bars, annual["Revenue"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
             f"{val/1e9:.2f}B", ha="center", va="bottom", fontsize=8)

# Trục y thứ hai: biên lợi nhuận %
ax2 = ax1.twinx()
ax2.plot(annual["Year"], annual["Margin"], color="crimson", marker="o",
         linewidth=2, markersize=5, label="Biên lợi nhuận %", zorder=3)
ax2.set_ylabel("Biên lợi nhuận (%)", color="crimson")
ax2.tick_params(axis="y", labelcolor="crimson")
ax2.set_ylim(0, 30)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax2.legend(loc="upper right")

# Chú thích
peak_rev = annual.loc[annual["Year"] == 2016, "Revenue"].values[0] / 1e9
ax1.annotate("Đỉnh: 2.10B",
             xy=(2016, peak_rev),
             xytext=(2013.8, 2.45),
             arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.5),
             fontsize=10, color="steelblue", fontweight="bold")

shift_rev = annual.loc[annual["Year"] == 2019, "Revenue"].values[0] / 1e9
ax1.annotate("Chuyển chế độ:\n−38.6% so CKN",
             xy=(2019, shift_rev),
             xytext=(2019.3, 1.75),
             arrowprops=dict(arrowstyle="->", color="dimgray", lw=1.5),
             fontsize=10, color="dimgray", fontweight="bold")

savefig("v2_1a_annual_revenue_era.png")


In [ ]:
# ── 1b: Doanh thu trên khách hàng tích lũy theo năm ────────────────────────────
customers = pd.read_csv(f"{DATA}customers.csv", parse_dates=["signup_date"])
customers["signup_year"] = customers["signup_date"].dt.year

cum_cust = (
    customers[customers["signup_year"] <= 2022]
    .groupby("signup_year").size()
    .sort_index()
    .cumsum()
    .reset_index(name="cum_customers")
    .rename(columns={"signup_year": "Year"})
)

rpc = annual[["Year", "Revenue"]].merge(cum_cust, on="Year")
rpc["rev_per_customer"] = rpc["Revenue"] / rpc["cum_customers"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rpc["Year"], rpc["rev_per_customer"] / 1e3,
        color="steelblue", marker="o", linewidth=2.5, markersize=7, zorder=3)
ax.fill_between(rpc["Year"], rpc["rev_per_customer"] / 1e3, alpha=0.12, color="steelblue")
ax.set_xlabel("Năm")
ax.set_ylabel("Doanh thu/Khách hàng tích lũy (Nghìn VND)")
ax.set_title("Doanh thu/Khách hàng sụp đổ dù số khách tăng mạnh")
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}K"))

start = rpc.iloc[0]
end   = rpc.iloc[-1]
ax.annotate("775K VND (2012)",
            xy=(start["Year"], start["rev_per_customer"] / 1e3),
            xytext=(2013.2, 620),
            arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.5),
            fontsize=10, fontweight="bold", color="steelblue")
ax.annotate("9.6K VND (2022)",
            xy=(end["Year"], end["rev_per_customer"] / 1e3),
            xytext=(2019.8, 90),
            arrowprops=dict(arrowstyle="->", color="dimgray", lw=1.5),
            fontsize=10, fontweight="bold", color="dimgray")

savefig("v2_1b_rev_per_customer.png")
print(rpc[["Year", "cum_customers", "rev_per_customer"]]
      .assign(rev_per_customer_K=lambda d: (d["rev_per_customer"] / 1e3).round(1))
      .to_string(index=False))


### Phát hiện Hồi 1 — Suy thoái

**Mô tả**
- Doanh thu hàng năm đạt đỉnh **2.10 tỷ VND vào năm 2016**, sau đó giảm xuống **1.17 tỷ VND vào năm 2022** — giảm 44.3% trong 6 năm.
- Mức giảm mạnh nhất trong một năm là **2018→2019: −38.6% so cùng kỳ năm trước** (1.85 tỷ → 1.14 tỷ), đánh dấu sự thay đổi chế độ rõ ràng.
- Tổng số khách hàng tích lũy tăng từ **957 (2012) lên 121.930 (2022)** — tăng 127 lần — trong khi doanh thu đình trệ.
- **Doanh thu/khách hàng tích lũy giảm 98.7%**: từ 775 nghìn VND (2012) xuống còn 9.6 nghìn VND (2022).

**Chẩn đoán**
- Số khách hàng tăng 22 lần nhưng doanh thu/khách hàng giảm 98.7% — doanh nghiệp đang thu hút khách hàng nhưng không khai thác được giá trị từ họ. Tăng trưởng về số lượng đã che khuất hoàn toàn sự thất bại trong việc sinh lời.
- Sự xói mòn doanh thu sau năm 2016 mang tính cơ cấu, không phải chu kỳ: ba năm suy giảm liên tiếp (2017–2019) tiếp theo là bình ổn cho thấy mức trần cầu vĩnh viễn, không phải sụt giảm tạm thời.
- Sự thay đổi chế độ vào 2018–2019 có thể phản ánh sự bão hòa trong phân khúc Streetwear cốt lõi (80% doanh thu), nơi thị trường mục tiêu thế hệ millennial đô thị đã đạt trần.

**Khuyến nghị**
- **Giai đoạn bình ổn 2019–2022 là điểm neo đúng cho dự báo 2023–2024. Không sử dụng xu hướng tăng trưởng 2012–2016. Tăng trọng số cho các năm gần đây 3× trong huấn luyện mô hình.**
- Mã hóa cờ nhị phân `post_2018` làm đặc trưng mô hình để ngăn dữ liệu giai đoạn tăng trưởng kéo dự đoán lên cao.
- Không nên kỳ vọng doanh thu/khách hàng phục hồi — mô hình hóa giai đoạn bình ổn là trạng thái ổn định, không phải quỹ đạo phục hồi.

---
## Hồi 2 — Bộ máy mùa vụ

In [ ]:
# ── Chuẩn bị dữ liệu Hồi 2 ─────────────────────────────────────────────────────────
plateau = sales[sales["Year"].between(2019, 2022)].copy()
plateau["Month"]  = plateau["Date"].dt.month
plateau["DOW"]    = plateau["Date"].dt.dayofweek   # 0 = Thứ Hai … 6 = Chủ Nhật
plateau["Margin"] = (plateau["Revenue"] - plateau["COGS"]) / plateau["Revenue"] * 100

MONTH_NAMES = [
    "T1", "T2", "T3", "T4", "T5", "T6",
    "T7", "T8", "T9", "T10", "T11", "T12",
]

print(f"Số dòng bình ổn : {len(plateau)}")
print(f"Khoảng thời gian: {plateau['Date'].min().date()} → {plateau['Date'].max().date()}")
print(plateau[["Revenue", "COGS", "Margin"]].describe().round(2))


In [ ]:
# ── 2a: Bản đồ nhiệt doanh thu theo tháng — tất cả các năm ──────────────────────────────────
from matplotlib.patches import Rectangle

monthly_rev = (
    sales.groupby(["Year", "Month"])["Revenue"].sum().reset_index()
)
hm = monthly_rev.pivot(index="Year", columns="Month", values="Revenue") / 1e9
hm.columns = MONTH_NAMES

# Nhãn chú thích (Tỷ VND)
annot_arr = np.array([[f"{v:.2f}B" for v in row] for row in hm.values])

fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(
    hm,
    cmap="YlOrRd",
    annot=annot_arr,
    fmt="",
    linewidths=0.5,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Doanh thu (Tỷ VND)"},
    annot_kws={"fontsize": 8},
)
ax.set_title("Bản đồ nhiệt doanh thu theo tháng — Tất cả các năm (Tỷ VND)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Tháng")
ax.set_ylabel("Năm")

# Viền đậm quanh các cột T4–T6 (chỉ số 0: T4=3, T5=4, T6=5 → x=3..6)
n_rows = len(hm)
border = Rectangle(
    (3, 0), 3, n_rows,
    fill=False, edgecolor="black", lw=3, clip_on=False, zorder=5,
)
ax.add_patch(border)

savefig("v2_2a_monthly_heatmap.png")


In [ ]:
# ── 2b: Doanh thu TB/ngày theo tháng + biên lợi nhuận % (giai đoạn bình ổn) ──────────
monthly_stats = (
    plateau.groupby("Month")
           .agg(avg_daily_rev=("Revenue", "mean"),
                avg_margin=("Margin",  "mean"))
           .reset_index()
)
monthly_stats["Month_name"] = [MONTH_NAMES[m - 1] for m in monthly_stats["Month"]]
plateau_mean_daily = plateau["Revenue"].mean()

x   = np.arange(12)
bw  = 0.35   # độ rộng nửa cột mỗi chuỗi

fig, ax1 = plt.subplots(figsize=(14, 6))

# ── Cột doanh thu trên trục trái ──
rev_x = x - 0.2
ax1.bar(rev_x, monthly_stats["avg_daily_rev"] / 1e6, width=bw,
        color="steelblue", alpha=0.82, zorder=2, label="Doanh thu TB/ngày")
ax1.axhline(plateau_mean_daily / 1e6, color="dimgray", linestyle="--", lw=1.6, zorder=3,
            label=f"TB bình ổn  {plateau_mean_daily / 1e6:.2f}M/ngày")
ax1.set_xlabel("Tháng")
ax1.set_ylabel("Doanh thu TB/ngày (Triệu VND)")
ax1.set_title("Doanh thu TB/ngày & Biên lợi nhuận theo Tháng — Giai đoạn bình ổn 2019–2022",
              fontsize=12, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(monthly_stats["Month_name"])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}M"))

# Gán nhãn cột Tháng 5 "2.9x Tháng 12"
may_val = monthly_stats.loc[monthly_stats["Month"] == 5, "avg_daily_rev"].values[0]
ax1.text(rev_x[4], may_val / 1e6 + 0.10,
         "2.9x\nTháng 12", ha="center", va="bottom",
         fontsize=8.5, fontweight="bold", color="steelblue")

# ── Cột biên lợi nhuận trên trục phải ──
ax2 = ax1.twinx()
mar_x = x + 0.2
ax2.bar(mar_x, monthly_stats["avg_margin"], width=bw * 0.75,
        color="crimson", alpha=0.48, zorder=2, label="Biên lợi nhuận TB %")
ax2.set_ylabel("Biên lợi nhuận TB (%)", color="crimson")
ax2.tick_params(axis="y", labelcolor="crimson")
ax2.set_ylim(0, 32)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))

# Hộp chú thích Q2 / Q3
q2_margin = monthly_stats[monthly_stats["Month"].between(4, 6)]["avg_margin"].mean()
q3_margin = monthly_stats[monthly_stats["Month"].between(7, 9)]["avg_margin"].mean()
q2_cx = np.mean(mar_x[[3, 4, 5]])
q3_cx = np.mean(mar_x[[6, 7, 8]])
box_kw = dict(boxstyle="round,pad=0.3", fc="white", ec="crimson", alpha=0.85)
ax2.text(q2_cx, q2_margin + 5, f"Q2: {q2_margin:.1f}%",
         ha="center", fontsize=9, color="crimson", fontweight="bold", bbox=box_kw)
ax2.text(q3_cx, q3_margin + 5, f"Q3: {q3_margin:.1f}%",
         ha="center", fontsize=9, color="crimson", fontweight="bold", bbox=box_kw)

# Chú giải tổng hợp
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8.5)

savefig("v2_2b_monthly_avg_rev.png")


In [ ]:
# ── 2c: Doanh thu trung bình theo ngày trong tuần ──────────────────────────────────────────
from matplotlib.patches import Patch

daily_dow = (
    plateau.groupby("DOW")["Revenue"]
           .mean()
           .reset_index()
           .rename(columns={"Revenue": "avg_rev"})
)
DOW_NAMES = ["T2", "T3", "T4", "T5", "T6", "T7", "CN"]
daily_dow["DOW_name"] = [DOW_NAMES[d] for d in daily_dow["DOW"]]
dow_colors = ["steelblue" if d < 5 else "salmon" for d in daily_dow["DOW"]]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(daily_dow["DOW_name"], daily_dow["avg_rev"] / 1e6,
       color=dow_colors, alpha=0.85, zorder=2)
ax.set_xlabel("Ngày trong tuần")
ax.set_ylabel("Doanh thu TB/ngày (Triệu VND)")
ax.set_title("Doanh thu TB/ngày theo Ngày trong tuần — Giai đoạn bình ổn 2019–2022",
             fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.2f}M"))

# Gán nhãn cột Thứ Tư
wed_val = daily_dow.loc[daily_dow["DOW"] == 2, "avg_rev"].values[0]
ax.text(2, wed_val / 1e6 + 0.03,
        "đỉnh\n+15.5% so Thứ Sáu", ha="center", va="bottom",
        fontsize=8.5, fontweight="bold", color="steelblue")

ax.legend(handles=[
    Patch(facecolor="steelblue", alpha=0.85, label="Ngày thường"),
    Patch(facecolor="salmon",    alpha=0.85, label="Cuối tuần"),
], loc="upper right")

savefig("v2_2c_dow_avg_rev.png")


### Phát hiện Hồi 2 — Bộ máy mùa vụ

**Mô tả**
- **Tháng 5 dẫn đầu**: trung bình **4.51 triệu VND/ngày** so với 1.57 triệu VND/ngày của Tháng 12 — tỷ lệ **2.9 lần**.
- Đỉnh Q2 (T4–T6) và đáy Q3 (T7–T9) định hình nhịp mùa vụ trong giai đoạn bình ổn (2019–2022).
- **Thứ Tư** là ngày trong tuần có doanh thu cao nhất, cao hơn **+15.5% so với Thứ Sáu**.

**Chẩn đoán**
- Đỉnh Q2 là **cầu thực sự**, KHÔNG được thúc đẩy bởi khuyến mãi — **biên lợi nhuận Q2 là 17.2%** so với 7.5% của Q3 xác nhận không có chiết khấu mạnh trong mùa cao điểm. Đỉnh do khuyến mãi sẽ nén biên lợi nhuận; thực tế ngược lại.
- Chu kỳ tuần (đỉnh Thứ Tư → đáy Thứ Sáu) cho thấy ý định mua hàng vào giữa tuần, có thể liên quan đến chu kỳ nhận lương.

**Khuyến nghị**
- **Tích trữ hàng tồn kho trước Tháng 4–6.** Tỷ lệ hết hàng 10% trong khoảng thời gian này gây thiệt hại khoảng **45 triệu VND doanh thu mỗi ngày.**
- **Chuyển chi tiêu khuyến mãi từ Q2 sang Q1/Q3.** Q2 tự bán được — giảm giá ở đây phá hủy biên lợi nhuận mà không tạo thêm doanh số.
- Để dự báo: mã hóa `month`, `day_of_week`, và `Q2_flag` (T4–T6 = 1) làm đặc trưng mô hình; các biến này giải thích thành phần phương sai lớn nhất trong giai đoạn bình ổn.

---
## Hồi 3 — Rò rỉ biên lợi nhuận

---
## Hồi 4 — Điểm neo dự báo